In [40]:
!pip install cloudscraper
import cloudscraper
import urllib3
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from time import sleep
from time import time
from google.colab import files
import requests
from bs4 import Comment
from bs4.element import Tag
header: str = "https://www.sports-reference.com/"

def soupify(url: str):

    http = urllib3.PoolManager()
    response = http.request('GET', url)
    soup = BeautifulSoup(response.data)

    return soup

def comment_helper(soup, obj: str):

    for comment in soup.find_all(string=lambda string: isinstance(string, Comment)):
        if comment.find(f"<{obj}") > 0: # and comment.find("totals_stats_sh") > 0:
            comment_soup = BeautifulSoup(comment, 'html.parser')
            res = comment_soup.find(obj)

    return res

def comment_helper(soup, obj: str, id = None):

    for comment in soup.find_all(string=lambda string: isinstance(string, Comment)):
        if not id:
            if comment.find(f"<{obj}") > 0:
                comment_soup = BeautifulSoup(comment, 'html.parser')
                res = comment_soup.find(obj)
        else:
            if comment.find(f"<{obj}") > 0 and comment.find(id) > 0:
                comment_soup = BeautifulSoup(comment, 'html.parser')
                res = comment_soup.find(obj)

    return res

def soupify_with_cloudscraper(url):
    scraper = cloudscraper.create_scraper()
    response = scraper.get(url)
    return BeautifulSoup(response.text, 'html.parser')

def get_table_ids(soup: "BeautifulSoup") -> list[str]:

    return list(t.get('id') for t in soup.find_all('table') if t.get('id'))

def get_columns(table_soup: "BeautifulSoup") -> list[str]:

    return list(t.get('data-stat') for t in table_soup.find('thead').find_all('th'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 4.6 MB/s eta 0:00:00


In [ ]:
# https://web.archive.org/web/20180531115621/https://www.pro-football-reference.com/blog/index4837.html?p=37
# https://www.basketball-reference.com/blog/indexba52.html?p=39

## pointless

In [ ]:
url: str = "https://www.sports-reference.com/cbb/seasons/schedule.cgi?month=11&day=3&year=2025"

In [ ]:
s = soupify_with_cloudscraper(url)

In [ ]:
def get_teams(span_obj: "Tag") -> tuple[str, str]:

    return list(a.contents[0] for a in span_obj.find_all('a'))

a, b = get_teams(spans[0])

'Florida'

In [ ]:
# spans = s.find_all('span')
# next(spans.index(sp) for sp in spans if not sp.get('class') and not sp.get('itemprop'))
spans = list(sp for sp in s.find_all('span') if not sp.get('class') and not sp.get('itemprop'))
# div = next(d for d in s.find_all('div') if d.find('h3'))
# div.find_all('span')[-1]
schedule: dict[str, dict] = {}
for game in spans:
    if "m" in game.find('span').contents[0]:
        if len(game.find_all('a')) > 1:
            teams = get_teams(game)
            for team in teams:
                if team in schedule.keys():
                    ...

## scraping stats leaderboard one team at a time

In [ ]:
header: str = "https://www.sports-reference.com"

In [ ]:
s = soupify_with_cloudscraper("https://www.sports-reference.com/cbb/seasons/men/2026-school-stats.html")

In [ ]:
table = s.find('table', {"id" : "basic_school_stats"})

In [ ]:
# year: int = 2026
# max_n = 40
# df = pd.DataFrame(columns = ["team", "year", "MoV"] + list(f"opp_{i}" for i in range(1,max_n)))
start = time()
for year in range(2022,2026):
    if year != 2020:
        s = soupify_with_cloudscraper(f"https://www.sports-reference.com/cbb/seasons/men/{year}-school-stats.html")
        table = s.find('table', {"id" : "basic_school_stats"})
        sleep(5)
        schedule = dict()
        # row = table.find('tbody').find('tr')
        for row in table.find('tbody').find_all('tr'):
            if row.find('td'):
                if row.find('td', {"data-stat" : "srs"}).contents:
                  team_name = row.find('a').contents[0]
                  team_url = (header + row.find('a')["href"]).replace(f"/{year}",f"/{year}-schedule")
                  team_sched = soupify_with_cloudscraper(team_url)
                  team_sched = team_sched.find('table', {"id" : "schedule"})
                  schedule[team_name] = {"opponents" : [], "point_diff" : []}
                  for game in team_sched.find('tbody').find_all('tr'): # .find('td', {"data-stat" : "opp_name"}).find('a')
                      if game.find('td'):
                          if game.find("td", {"data-stat" : "date_game"}).find('a'):
                              if game.find('td', {"data-stat" : "opp_name"}).find('a') and game.find('td', {"data-stat" : "srs"}).contents:
                                  if game.find('td', {"data-stat" : "game_type"}).contents[0] in {"REG", "CTOURN"}:
                                      schedule[team_name]["opponents"].append(game.find('td', {"data-stat" : "opp_name"}).find('a').contents[0])
                                      ps = int(game.find('td', {"data-stat" : "pts"}).contents[0])
                                      pa = int(game.find('td', {"data-stat" : "opp_pts"}).contents[0])
                                      schedule[team_name]["point_diff"].append(ps - pa)
                  sleep(5)
        max_n = max({len(schedule[team]["opponents"]) for team in schedule})
        df = pd.DataFrame(columns = ["team", "year", "MoV"] + list(f"opp_{i}" for i in range(1,max_n + 1)))
        for team in schedule:
            n = len(schedule[team]["opponents"])
            # df.loc[len(df.index)] = [team] + [np.mean(schedule[team]["point_diff"])] + schedule[team]["opponents"] + [0]*(max_n - n)
            df.loc[len(df.index)] = [team, year] + [np.mean(schedule[team]["point_diff"])] + schedule[team]["opponents"] + [0]*(max_n - n)
        print(f"{year} done: {time() - start}")
        df.to_csv(f"schedule{year-2000}DF.csv", index = False)
        files.download(f"schedule{year-2000}DF.csv")

2022 done: 1856.5234208106995


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

2023 done: 3740.6119940280914


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

2025 done: 7522.599548339844


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df.to_csv("scheduleDF.csv", index = False)
files.download("scheduleDF")

In [ ]:
schedule[team_name] = {"opponents" : [], "point_diff" : []}
for game in team_sched.find('tbody').find_all('tr'): # .find('td', {"data-stat" : "opp_name"}).find('a')
    if game.find('td'):
        if game.find("td", {"data-stat" : "date_game"}).find('a'):
            if game.find('td', {"data-stat" : "opp_name"}).find('a'):
                if game.find('td', {"data-stat" : "game_type"}).contents[0] in {"REG", "CTOURN"}:
                    schedule[team_name]["opponents"].append(game.find('td', {"data-stat" : "opp_name"}).find('a').contents[0])
                    ps = int(game.find('td', {"data-stat" : "pts"}).contents[0])
                    pa = int(game.find('td', {"data-stat" : "opp_pts"}).contents[0])
                    schedule[team_name]["point_diff"].append(ps - pa)

In [ ]:
df = pd.DataFrame(columns = ["team", "MoV"] + list(f"opp_{i}" for i in range(1,36)))
for team in schedule:
    n = len(schedule[team]["opponents"])
    df.loc[len(df.index)] = [team] + [np.mean(schedule[team]["point_diff"])] + schedule[team]["opponents"] + [0]*(35 - n)

In [ ]:
df.to_csv("scheduleDf26.csv", index = False)

# calculating SRS and SOS

In [1]:
import pandas as pd
import numpy as np

In [ ]:
teams = len(schedule)
ids = {list(schedule.keys())[i]:i for i in range(teams)}

In [ ]:
# 'Southern Illinois-Edwardsville' vs "SIU Edwardsville"
ids['Southern Illinois-Edwardsville'] = ids["SIU Edwardsville"]
# 'Virginia Military Institute' vs "VMI"
ids["Virginia Military Institute"] = ids["VMI"]

In [ ]:
# 365 x 366 matrix, column 1 is the team itself, column 2 is the mov, columns 3-367 are the team ratings, each row is a team rating
# M = np.zeros(shape = (365,366))
# M = np.zeros(shape = (365,365))
M = np.zeros(shape = (teams,teams))
# M[:,0] = 1
# M[:,1] = df.iloc[:,1]
# M[:,0] = df.iloc[:,1]
for team in schedule:
    n = len(schedule[team]["opponents"])
    M[ids[team], ids[team]] = -1
    for opp in schedule[team]["opponents"]:
        M[ids[team], ids[opp]] += 1/n
M[-1,:] = 1

In [ ]:
from scipy import linalg, sparse
# M = np.zeros(shape = (365,365))
M = np.zeros(shape = (teams,teams))
for team in schedule:
    n = len(schedule[team]["opponents"])
    M[ids[team], ids[team]] = -1
    for opp in schedule[team]["opponents"]:
        M[ids[team], ids[opp]] += 1/n
M[-1,:] = 1 # adjusting

A = np.array(-df.iloc[:,2])
A[-1] = 0 # adjusting
opp_ratings = pd.DataFrame(columns = ["team", "id", "rating"])
M_sparse = sparse.csc_matrix(M)
opp_ratings["team"] = df['team']
# opp_ratings["id"] = range(365)
opp_ratings["id"] = range(teams)
opp_ratings["rating"] = sparse.linalg.spsolve(M_sparse, A)
# linalg.solve(M, A)
opp_ratings

,team,id,rating
0,Abilene Christian,0,-5.048652
1,Air Force,1,-7.286466
2,Akron,2,3.953288
3,Alabama,3,26.358786
4,Alabama A&M,4,-19.964784
...,...,...,...
359,Wright State,359,-4.493225
360,Wyoming,360,0.174830
361,Xavier,361,15.947329
362,Yale,362,7.165935


In [ ]:
curr = "Akron"
sos = 0
n = len(schedule[curr]["opponents"])
mov = np.mean(schedule[curr]["point_diff"])
for opp in schedule[curr]["opponents"]:
    sos += opp_ratings.loc[opp_ratings["id"] == ids[opp],"rating"].values[0]
sos/n #+ mov

np.float64(-4.109212270335497)

### from gemini

In [ ]:
# 1. Create an "Averaging Matrix" by clearing out the -1 from the diagonal
M_opponents = M.copy()
np.fill_diagonal(M_opponents, 0)

# 2. Extract the solved ratings as a flat NumPy array (aligned by ID)
ratings_vector = opp_ratings.sort_values("id")["rating"].to_numpy()

# 3. Matrix multiply: This calculates the mean opponent rating for every team at once
# (Matrix rows sum to 1, multiplying each opponent weight by their rating)
sos_vector = M_opponents.dot(ratings_vector)

# 4. Map the SOS array directly back into your DataFrame
opp_ratings["SOS"] = opp_ratings["id"].map(lambda x: sos_vector[x])

print(opp_ratings.sort_values(by="rating", ascending=False))

                         team   id     rating        SOS
175                  Michigan  175  32.893065  15.275418
73                       Duke   73  31.969284  12.822225
10                    Arizona   10  30.336409  12.983468
86                    Florida   86  28.316134  13.528255
126                Iowa State  126  27.566586  10.919527
..                        ...  ...        ...        ...
95               Gardner-Webb   95 -21.947145  -1.447145
67             Delaware State   67 -22.139629 -10.398888
352          Western Illinois  352 -23.028324  -7.742609
60               Coppin State   60 -24.425668  -7.692334
182  Mississippi Valley State  182 -27.941756  -7.473006

[365 rows x 4 columns]


# Looping

In [5]:
import pandas as pd
import numpy as np
import scipy
from scipy import linalg, sparse

In [3]:
year: int = 10
df = pd.read_csv("schedule10DF.csv")
teams = len(df["team"])
ids = {list(df["team"])[i]:i for i in range(teams)}

In [4]:
# ids['Southern Illinois-Edwardsville'] = ids["SIU Edwardsville"]
ids["Virginia Military Institute"] = ids["VMI"]

In [5]:
schedule = dict()
for team in list(df["team"]):
    schedule[team] = dict()
    id = ids[team]
    L = list(df.iloc[id][[c for c in df.columns if "opp" in c]].values)
    L = list(team for team in L if team != "0")
    schedule[team]["opponents"] = L
    schedule[team]["MoV"] = df.iloc[id]["MoV"]

In [6]:
from scipy import linalg, sparse
# M = np.zeros(shape = (365,365))
M = np.zeros(shape = (teams,teams))
for team in schedule:
    n = len(schedule[team]["opponents"])
    M[ids[team], ids[team]] = -1
    for opp in schedule[team]["opponents"]:
        M[ids[team], ids[opp]] += 1/n
# M[-1,:] = 1 # adjusting

A = np.array(-df.iloc[:,2])
# A[-1] = 0 # adjusting
opp_ratings = pd.DataFrame(columns = ["team", "id", "rating"])
M_sparse = sparse.csc_matrix(M)
opp_ratings["team"] = df['team']
# opp_ratings["id"] = range(365)
opp_ratings["id"] = range(teams)
# opp_ratings = pd.merge(opp_ratings, df[["team", "MoV"]])
opp_ratings["rating"] = sparse.linalg.spsolve(M_sparse, A)
# linalg.solve(M, A)

In [7]:
for team in list(df["team"]):
    # curr = "Akron"
    id = ids[team]
    schedule[team]["rating"] = opp_ratings.iloc[id]["rating"]
    sos = 0
    n = len(schedule[team]["opponents"])
    mov = np.mean(schedule[team]["MoV"])
    for opp in schedule[team]["opponents"]:
        sos += opp_ratings.loc[opp_ratings["id"] == ids[opp],"rating"].values[0]
    schedule[team]["SOS"] = sos/n #+ mov
    schedule[team]["SRS"] = schedule[team]["SOS"] + schedule[team]["MoV"]

In [ ]:
schedule['Baylor']

{'opponents': ['Norfolk State',
  'Hartford',
  'Southern',
  'Alabama',
  'Iona',
  'Xavier',
  'Arizona State',
  'Jackson State',
  'UT Arlington',
  'Arkansas',
  'South Carolina',
  'Morgan State',
  'Oklahoma',
  'Colorado',
  'Oklahoma State',
  'Kansas',
  'Massachusetts',
  'Kansas State',
  'Texas',
  'Iowa State',
  'Texas A&M',
  'Nebraska',
  'Missouri',
  'Texas Tech',
  'Oklahoma State',
  'Texas A&M',
  'Oklahoma',
  'Texas Tech',
  'Texas',
  'Texas',
  'Kansas State'],
 'MoV': np.float64(10.161290322580646),
 'rating': np.float64(18.432278794670292),
 'SOS': np.float64(8.270988472089652),
 'SRS': np.float64(18.432278794670296)}

In [3]:
final_df = pd.DataFrame(columns=["year", "team", "MoV", "SOS", "SRS"])
for team in list(df["team"]):
            # curr = "Akron"
            id = ids[team]
            schedule[team]["rating"] = opp_ratings.iloc[id]["rating"]
            sos = 0
            n = len(schedule[team]["opponents"])
            mov = np.mean(schedule[team]["MoV"])
            for opp in schedule[team]["opponents"]:
                sos += opp_ratings.loc[opp_ratings["id"] == ids[opp],"rating"].values[0]
            schedule[team]["SOS"] = sos/n #+ mov
            schedule[team]["SRS"] = schedule[team]["SOS"] + schedule[team]["MoV"]
            final_df.loc[len(final_df.index)] = [2010,
                                                team,
                                                schedule[team]["MoV"],
                                                schedule[team]["SOS"],
                                                schedule[team]["SRS"]]

NameError: name 'df' is not defined

In [6]:
final_df = pd.DataFrame(columns=["year", "team", "games", "MoV", "SOS", "SRS"])
# for year in range(10,27):
for year in range(10,16):
    if year != 20:
        if year != 26:
            df = pd.read_csv(f"schedule{year}DF.csv")
        else:
            df = pd.read_csv(f"scheduleDf26.csv")
        teams = len(df["team"])
        ids = {list(df["team"])[i]:i for i in range(teams)}

        if "SIU Edwardsville" in ids:
            ids['Southern Illinois-Edwardsville'] = ids["SIU Edwardsville"]
        if "VMI" in ids:
            ids["Virginia Military Institute"] = ids["VMI"]

        schedule = dict()
        for team in list(df["team"]):
            schedule[team] = dict()
            id = ids[team]
            L = list(df.iloc[id][[c for c in df.columns if "opp" in c]].values)
            L = list(team for team in L if team != "0")
            schedule[team]["opponents"] = L
            schedule[team]["MoV"] = df.iloc[id]["MoV"]

        M = np.zeros(shape = (teams,teams))
        for team in schedule:
            n = len(schedule[team]["opponents"])
            M[ids[team], ids[team]] = -1
            for opp in schedule[team]["opponents"]:
                M[ids[team], ids[opp]] += 1/n
        M[-1,:] = 1 # adjusting

        A = np.array(-df.iloc[:,2]) if year != 26 else np.array(-df.iloc[:,1])
        A[-1] = 0 # adjusting
        opp_ratings = pd.DataFrame(columns = ["team", "id", "rating"])
        M_sparse = sparse.csc_matrix(M)
        opp_ratings["team"] = df['team']
        # opp_ratings["id"] = range(365)
        opp_ratings["id"] = range(teams)
        # opp_ratings = pd.merge(opp_ratings, df[["team", "MoV"]])
        opp_ratings["rating"] = sparse.linalg.spsolve(M_sparse, A)

        for team in list(df["team"]):
            # curr = "Akron"
            id = ids[team]
            schedule[team]["rating"] = opp_ratings.iloc[id]["rating"]
            sos = 0
            n = len(schedule[team]["opponents"])
            mov = np.mean(schedule[team]["MoV"])
            for opp in schedule[team]["opponents"]:
                sos += opp_ratings.loc[opp_ratings["id"] == ids[opp],"rating"].values[0]
            schedule[team]["SOS"] = sos/n #+ mov
            schedule[team]["SRS"] = schedule[team]["SOS"] + schedule[team]["MoV"]
            final_df.loc[len(final_df.index)] = [2000 + year,
                                                team,
                                                 n,
                                                schedule[team]["MoV"],
                                                schedule[team]["SOS"],
                                                schedule[team]["SRS"]]

In [8]:
final_df.to_csv("ncaa_srs_sos_1015.csv") #.sort_values(["year","SRS"], ascending=[True, True]).groupby("year").head(3)

In [59]:
final_df.sort_values(["year","SRS"], ascending=[True, False]).groupby("year").head(3)

,year,team,games,MoV,SOS,SRS
123,2010,Kansas,33,17.000000,7.694047,24.694047
68,2010,Duke,33,16.636364,7.840241,24.476604
270,2010,Syracuse,32,14.593750,6.404727,20.998477
546,2011,Ohio State,34,17.264706,-0.360968,16.903738
405,2011,Duke,34,16.794118,-0.266368,16.527750
462,2011,Kansas,34,17.147059,-0.724634,16.422425
811,2012,Kentucky,34,17.676471,37.520248,55.196719
890,2012,Ohio State,34,15.705882,39.296189,55.002071
845,2012,Michigan State,33,12.151515,40.710767,52.862282
1140,2013,Indiana,33,17.545455,12.526016,30.071470


In [28]:
final_df.sort_values(["year","SRS"], ascending=[True, False])

,year,team,games,MoV,SOS,SRS
123,2010,Kansas,33,17.000000,7.694047,24.694047
68,2010,Duke,33,16.636364,7.840241,24.476604
270,2010,Syracuse,32,14.593750,6.404727,20.998477
128,2010,Kentucky,34,13.970588,5.660489,19.631078
125,2010,Kansas State,31,9.870968,9.311339,19.182307
...,...,...,...,...,...,...
1728,2015,Alcorn State,30,-10.800000,-15.463779,-26.263779
1762,2015,Central Arkansas,29,-16.896552,-11.376067,-28.272618
1897,2015,Mississippi Valley State,32,-16.156250,-13.177848,-29.334098
1805,2015,Florida A&M,29,-17.517241,-13.964068,-31.481310


In [ ]:
final_df.to_csv("SRS_SOS_fixed.csv",index = False)

# Iterative method

In [39]:
temp = final_df.loc[final_df["year"] == 2010,["games", "MoV","SRS"]].copy()
(temp["games"]*temp["SRS"]).sum()

np.float64(-10450.45330619963)

In [55]:
s = soupify_with_cloudscraper("https://www.sports-reference.com/cbb/seasons/men/2025-school-stats.html")

In [57]:
# table = s.find("table")
sum(float(t.find('td', {"data-stat" : "g"}).contents[0])*float(t.find('td', {"data-stat" : "srs"}).contents[0]) for t in table.find_all('tr') if t.find('td') and t.find('td', {"data-stat" : "srs"}).contents)

-3101.0100000000007

In [30]:
schedule['Abilene Christian']

{'opponents': ['Tulsa',
  'Duquesne',
  'UC Riverside',
  'Sacramento State',
  'Houston',
  'Loyola (IL)',
  'Boise State',
  'South Carolina State',
  'Arkansas-Pine Bluff',
  'Grand Canyon',
  'Central Arkansas',
  'Nicholls State',
  'Northwestern State',
  'Stephen F. Austin',
  'McNeese',
  'Incarnate Word',
  'Sam Houston',
  'Lamar',
  'Houston Christian',
  'Southeastern Louisiana',
  'Lamar',
  'Sam Houston',
  'Southeastern Louisiana',
  'Texas A&M-Corpus Christi',
  'New Orleans',
  'Houston Christian',
  'Incarnate Word',
  'Texas A&M-Corpus Christi'],
 'MoV': np.float64(-10.857142857142858),
 'rating': np.float64(-22.605838350901458),
 'SOS': np.float64(-11.748695493758618),
 'SRS': np.float64(-22.605838350901475)}